In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import kagglehub
import warnings
warnings.filterwarnings('ignore')
#!pip install --upgrade kagglehub
#!pip install --upgrade numpy scipy scikit-learn
#pip install numpy scipy scikit-learn pandas nltk matplotlib gensim
#!pip install kagglehub
import re
import string
import os
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from gensim.models import Word2Vec
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.naive_bayes import GaussianNB
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\bhanu\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\bhanu\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\bhanu\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

# 2. LOAD DATASET

In [2]:
# 2. LOAD DATASET
df = pd.read_csv(r"C:\Users\bhanu\Downloads\archive file\twitter_dataset.csv")   
# Select required columns
df.columns = ["id", "entity", "sentiment", "text"]
# Remove missing values
df.dropna(inplace=True)
df

,id,entity,sentiment,text
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...
...,...,...,...,...
74676,9200,Nvidia,Positive,Just realized that the Windows partition of my...
74677,9200,Nvidia,Positive,Just realized that my Mac window partition is ...
74678,9200,Nvidia,Positive,Just realized the windows partition of my Mac ...
74679,9200,Nvidia,Positive,Just realized between the windows partition of...


In [3]:
df.shape

(73995, 4)

In [4]:
print("Shape:", df.shape)
print(df['sentiment'].value_counts())
print(df.isnull().sum())

Shape: (73995, 4)
sentiment
Negative      22358
Positive      20654
Neutral       18108
Irrelevant    12875
Name: count, dtype: int64
id           0
entity       0
sentiment    0
text         0
dtype: int64


In [5]:
df['sentiment'].value_counts()


sentiment
Negative      22358
Positive      20654
Neutral       18108
Irrelevant    12875
Name: count, dtype: int64

# 3. PREPROCESSING FUNCTION

In [21]:
# 3. PREPROCESSING FUNCTION
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    
    # It is used to handle non string
    if not isinstance(text, str):
        return ""
    
    # Lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # Remove mentions and hashtags
    text = re.sub(r'@\w+|#\w+', '', text)
    
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Keep only alphabets
    text = re.sub(r'[^a-z\s]', '', text)
    
    # Remove extra spaces
    text = text.strip()
    
    # Tokenization
    tokens = word_tokenize(text)
    
    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]
    
    # Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    return " ".join(tokens)
    
df['clean_text'] = df['text'].apply(preprocess_text)
df[['text', 'clean_text']].head()

,text,clean_text
0,I am coming to the borders and I will kill you...,coming border kill
1,im getting on borderlands and i will kill you ...,im getting borderland kill
2,im coming on borderlands and i will murder you...,im coming borderland murder
3,im getting on borderlands 2 and i will murder ...,im getting borderland murder
4,im getting into borderlands and i can murder y...,im getting borderland murder


In [7]:
df.sample(30)

,id,entity,sentiment,text,clean_text
22684,4286,CS-GO,Neutral,I,
65060,7943,MaddenNFL,Negative,Bro...this controller doesnt go thru the TV! @...,brothis controller doesnt go thru tv wtf
13244,8674,NBA2K,Negative,Hold onnnn. fw 2k it might be glitchy... tho,hold onnnn fw k might glitchy tho
10796,13057,Xbox(Xseries),Irrelevant,"Xbox head: E3 IS CANCELLED, WHAT ARE WE GONNA ...",xbox head e cancelled gon na get show big cons...
63696,7716,MaddenNFL,Neutral,@ EASPORTS _ MUT @ EAMaddenNFL @ EA _ KRAELO @...,easports mut eamaddennfl ea kraelo maddenbible...
32529,7580,LeagueOfLegends,Positive,by RhandlerR looking good on RhandlerR - don’t...,rhandlerr looking good rhandlerr dont forget r...
30839,7299,LeagueOfLegends,Irrelevant,Congratulations to our very own @JulienMid and...,congratulation selected team upcoming league l...
30130,779,ApexLegends,Negative,was,
45727,11849,Verizon,Negative,"@ Verizon @ yofGilroy, as soon as we leave tow...",verizon yofgilroy soon leave town g speed exce...
42303,10058,PlayerUnknownsBattlegrounds(PUBG),Negative,Just @PUBG is hi I m facing this a hand in gam...,hi facing hand game problem little game plan l...


In [8]:
# 4. TRAIN-TEST SPLIT
X = df['clean_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape)
print(X_test.shape)

(59196,)
(14799,)


In [9]:
# ----- Bag of Words -----
bow = CountVectorizer(max_features=5000)
X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)
print(X_train_bow.shape),
print(X_test_bow.shape)

(59196, 5000)
(14799, 5000)


In [10]:
# ----- TF-IDF -----
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)
print(X_train_tfidf.shape)
print(X_test_tfidf.shape)

(59196, 5000)
(14799, 5000)


# 6. MODEL BUILDING

In [11]:
# 6. MODEL BUILDING
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Naive Bayes": MultinomialNB(),
    "Decision Tree": DecisionTreeClassifier()}

# Logistic Regression

In [12]:
lr = LogisticRegression()
lr.fit(X_train_tfidf, y_train)

y_pred_lr = lr.predict(X_test_tfidf)

print("\n Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))


 Logistic Regression Accuracy: 0.6776133522535307
              precision    recall  f1-score   support

  Irrelevant       0.67      0.49      0.57      2575
    Negative       0.72      0.76      0.74      4472
     Neutral       0.62      0.65      0.64      3621
    Positive       0.68      0.72      0.70      4131

    accuracy                           0.68     14799
   macro avg       0.67      0.66      0.66     14799
weighted avg       0.68      0.68      0.67     14799



# Navi baias

In [13]:
nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)

y_pred_nb = nb.predict(X_test_tfidf)

print("\n Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))


 Naive Bayes Accuracy: 0.6380836543009663
              precision    recall  f1-score   support

  Irrelevant       0.75      0.34      0.47      2575
    Negative       0.62      0.81      0.70      4472
     Neutral       0.66      0.53      0.59      3621
    Positive       0.62      0.74      0.67      4131

    accuracy                           0.64     14799
   macro avg       0.66      0.60      0.61     14799
weighted avg       0.65      0.64      0.63     14799



# Decision tree

In [14]:
dt = DecisionTreeClassifier()
dt.fit(X_train_tfidf, y_train)

y_pred_dt = dt.predict(X_test_tfidf)

print("\nDecision Tree Accuracy:", accuracy_score(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt))
     


Decision Tree Accuracy: 0.7675518616122711
              precision    recall  f1-score   support

  Irrelevant       0.76      0.69      0.72      2575
    Negative       0.82      0.79      0.80      4472
     Neutral       0.76      0.74      0.75      3621
    Positive       0.73      0.82      0.77      4131

    accuracy                           0.77     14799
   macro avg       0.77      0.76      0.76     14799
weighted avg       0.77      0.77      0.77     14799



In [15]:
# 7. EVALUATION FUNCTION
def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    return acc, prec, rec, f1
results = []

for name, model in models.items():
    
    # Train on TF-IDF
    model.fit(X_train_tfidf, y_train)
    acc, prec, rec, f1 = evaluate(model, X_test_tfidf, y_test)
    
    results.append([name, "TF-IDF", acc, prec, rec, f1])
    
    # Train on BoW
    model.fit(X_train_bow, y_train)
    acc, prec, rec, f1 = evaluate(model, X_test_bow, y_test)
    
    results.append([name, "BoW", acc, prec, rec, f1])

# Convert results to DataFrame
results_df = pd.DataFrame(results, columns=[
    "Model", "Vectorizer", "Accuracy", "Precision", "Recall", "F1 Score"
])

print("\nModel Comparison:\n")
print(results_df)


Model Comparison:

                 Model Vectorizer  Accuracy  Precision    Recall  F1 Score
0  Logistic Regression     TF-IDF  0.680924   0.680426  0.680924  0.677904
1  Logistic Regression        BoW  0.697615   0.699016  0.697615  0.694865
2          Naive Bayes     TF-IDF  0.638084   0.652622  0.638084  0.625008
3          Naive Bayes        BoW  0.632745   0.631713  0.632745  0.627514
4        Decision Tree     TF-IDF  0.768768   0.770879  0.768768  0.768433
5        Decision Tree        BoW  0.797419   0.800004  0.797419  0.797542


# 9. BEST MODEL (Logistic Regression + TF-IDF)

In [16]:
best_model = LogisticRegression(max_iter=1000)
best_model.fit(X_train_tfidf, y_train)

y_pred = best_model.predict(X_test_tfidf)

print("\nBest Model Report:\n")
print(classification_report(y_test, y_pred))



Best Model Report:

              precision    recall  f1-score   support

  Irrelevant       0.68      0.50      0.57      2575
    Negative       0.72      0.77      0.75      4472
     Neutral       0.63      0.66      0.64      3621
    Positive       0.68      0.72      0.70      4131

    accuracy                           0.68     14799
   macro avg       0.68      0.66      0.67     14799
weighted avg       0.68      0.68      0.68     14799



# 10. TEST WITH NEW INPUT

In [24]:
sample = ["This pen is good"]

# FIXED LINE (important)
sample_clean = [preprocess_text(text) for text in sample]

sample_vec = tfidf.transform(sample_clean)

prediction = best_model.predict(sample_vec)

print("\nSample Prediction:", prediction)


Sample Prediction: ['Positive']


# Word 2 Vec

In [ ]:
# Convert sentences into list of words
from sklearn.naive_bayes import GaussianNB
nb_w2v = MultinomialNB()
from sklearn.naive_bayes import GaussianNB
nb_w2v = GaussianNB()

print("Accuracy:", accuracy_score(y_test, y_pred))
tokenized_text = df['clean_text'].apply(lambda x: x.split())
w2v_model = Word2Vec(
    sentences=tokenized_text,
    vector_size=100,   # vector size
    window=5,
    min_count=2,
    workers=4
)
import numpy as np

def get_avg_word2vec(words, model, vector_size):
    vec = np.zeros(vector_size)
    count = 0
    
    for word in words:
        if word in model.wv:
            vec += model.wv[word]
            count += 1
    
    if count != 0:
        vec = vec / count
    
    return vec

In [15]:
# create futher mix

In [16]:
X_w2v = np.array([
    get_avg_word2vec(words, w2v_model, 100)
    for words in tokenized_text
])

In [17]:
X_train_w2v, X_test_w2v, y_train, y_test = train_test_split(
    X_w2v, y, test_size=0.2, random_state=42, stratify=y)

In [18]:
# Logistic Regression
lr_w2v = LogisticRegression(max_iter=1000)
lr_w2v.fit(X_train_w2v, y_train)

# Naive Bayes (NOTE: works less effectively with dense data)
from sklearn.naive_bayes import GaussianNB

nb_w2v = GaussianNB()
nb_w2v.fit(X_train_w2v, y_train)

# Decision Tree
dt_w2v = DecisionTreeClassifier()
dt_w2v.fit(X_train_w2v, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [19]:
def evaluate_model(model, X_test, y_test, name):
    y_pred = model.predict(X_test)
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))

evaluate_model(lr_w2v, X_test_w2v, y_test, "LR + Word2Vec")
evaluate_model(nb_w2v, X_test_w2v, y_test, "NB + Word2Vec")
evaluate_model(dt_w2v, X_test_w2v, y_test, "DT + Word2Vec")


LR + Word2Vec
Accuracy: 0.5184809784444895

NB + Word2Vec
Accuracy: 0.440570308804649

DT + Word2Vec
Accuracy: 0.5932157578214744


# Clean Flow Diagram 
### Raw Text Data
      ↓
### Text Preprocessing
      ↓
### Feature Engineering (TF-IDF)
      ↓
### Model Training (LR, NB, DT)
      ↓
### Model Evaluation
      ↓
### Comparison & Insights

# Comparison & Insights
1. Best Preprocessing Steps
The following preprocessing steps gave the best results:

### Lowercasing → ensured uniform text format
### Removing URLs & special characters → removed noise
### Stopword removal → reduced unnecessary words
### Lemmatization → preserved meaning while reducing word forms

# 2. Best Vectorization Technique
#### ------> TF-IDF (Best)
Captures importance of words
Reduces effect of common words
Works very well with linear models
#### -------> Bag of Words
Simple but treats all words equally
Less effective than TF-IDF
#### --------> Word2Vec (Optional)
Captures semantic meaning
But requires more tuning and data

# 3.Best Model
### -----> Logistic Regression (Best)
Highest accuracy and F1 score
Works well with TF-IDF
Handles high-dimensional text data efficiently
### -----> Naive Bayes
Fast and simple
Good baseline model
### ------> Decision Tree
Lower performance
Prone to overfitting
### -------> Random Forest (Optional)
Better than Decision Tree
But slower and more complex

# Model Comparison Summary
### ---- Component-------> Best Choice
### -----Preprocessing--------> Lemmatization + Cleaning
### -----Vectorization------->TF-IDF
### -----Model------>Logistic Regression

# Final------>
+ ### The preprocessing steps, including lowercasing, removal of noise, stopwords, and lemmatization, significantly improved data quality. Among vectorization techniques, TF-IDF performed best by assigning importance to relevant words. Logistic Regression achieved the highest performance due to its efficiency in handling high-dimensional sparse data. Naive Bayes provided a strong baseline, while Decision Tree showed limitations due to overfitting. Overall, the combination of TF-IDF and Logistic Regression yielded the best results. Trade-offs between model complexity, interpretability, and performance were also observed.